# 05 · Construir predictores disponibles antes de la inspección
**CRISP-ML(Q) · Ingeniería de datos.** Fuente: Hansen et al. (2013), UMD/Google/USGS/NASA, Global Forest Change v1.12, CC BY 4.0 según el Hito 1. Se conservan la máscara terrestre, el umbral de cobertura inicial, la cuadrícula y la población elegible del RF. La versión v1.13 se reserva para verificar 2025 en el cuaderno 06.

Referencia: [Hansen et al., Science](https://doi.org/10.1126/science.1244693).

In [1]:
from pathlib import Path
import os
import sys
import pandas as pd
import numpy as np
from IPython.display import display

raiz = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'config/hito2.yaml').exists())
os.chdir(raiz)
sys.path.insert(0, str(raiz / 'scripts'))
import linea_base_multizona as base

cfg, huella = base.cargar_config('config/hito2.yaml')
zonas = cfg['zonas']
celdas = cfg['celdas']
salida = Path('outputs/ruta_espacial')
cache = Path('data/processed/ruta_espacial')
salida.mkdir(parents=True, exist_ok=True)
cache.mkdir(parents=True, exist_ok=True)

In [2]:
from src.forestguard.features import calcular_cubos, tabla_features, FEATURES
from scipy.ndimage import convolve, distance_transform_edt
from rasterio import open as abrir_raster

for zona, bbox in {**zonas['entrenamiento'], **zonas['retenidas']}.items():
    base.asegurar_recorte(zona, bbox, cfg['datos']['tesela'])
    perfiles = []
    for capa in base.CAPAS:
        with abrir_raster(base.PROC / f'{zona}_{capa}.tif') as src:
            perfiles.append((src.shape, src.transform, src.crs, tuple(src.bounds)))
    assert perfiles.count(perfiles[0]) == len(perfiles)
    assert np.allclose(perfiles[0][-1], bbox)
    print(zona, perfiles[0][0], perfiles[0][2])

guaviare (4000, 4000) EPSG:4326
[ya existe] Hansen_GFC-2024-v1.12_treecover2000_10N_080W.tif
[ya existe] Hansen_GFC-2024-v1.12_lossyear_10N_080W.tif
[ya existe] Hansen_GFC-2024-v1.12_datamask_10N_080W.tif


[recortado] macarena_treecover2000.tif (4000, 4000)


[recortado] macarena_lossyear.tif (4000, 4000)
[recortado] macarena_datamask.tif (4000, 4000)
macarena (4000, 4000) EPSG:4326
[ya existe] Hansen_GFC-2024-v1.12_treecover2000_10N_080W.tif
[ya existe] Hansen_GFC-2024-v1.12_lossyear_10N_080W.tif
[ya existe] Hansen_GFC-2024-v1.12_datamask_10N_080W.tif


[recortado] chaira_treecover2000.tif (4000, 4000)


[recortado] chaira_lossyear.tif (4000, 4000)
[recortado] chaira_datamask.tif (4000, 4000)
chaira (4000, 4000) EPSG:4326
[ya existe] Hansen_GFC-2024-v1.12_treecover2000_10N_080W.tif
[ya existe] Hansen_GFC-2024-v1.12_lossyear_10N_080W.tif
[ya existe] Hansen_GFC-2024-v1.12_datamask_10N_080W.tif


[recortado] putumayo_treecover2000.tif (4000, 4000)


[recortado] putumayo_lossyear.tif (4000, 4000)
[recortado] putumayo_datamask.tif (4000, 4000)
putumayo (4000, 4000) EPSG:4326


La distancia euclidiana se mide en celdas hasta la más cercana que perdió ≥2 ha durante T. Si el recuadro carece de focos se usa su diagonal. Los anillos cuadrados de radios 2 y 3 promedian hectáreas por celda vecina sobre el perímetro, corrigiendo el denominador en bordes. La tendencia es la pendiente en ha/año de las tres últimas observaciones. El recorte limita la información disponible en sus bordes.

In [3]:
def tabla_espacial(cubo, t):
    tabla = tabla_features(cubo, t, celdas['alerta_ha']).copy()
    historia = cubo['perdida'][t-2:t+1] * cubo['ha_por_pixel']
    ultimo = historia[-1]
    focos = ultimo >= celdas['alerta_ha']
    indices = (tabla.fila.to_numpy(), tabla.col.to_numpy())
    distancia = (distance_transform_edt(~focos) if focos.any()
                 else np.full(ultimo.shape, np.hypot(*ultimo.shape)))
    tabla['distancia_foco_celdas'] = distancia[indices]
    tabla['tendencia_ha_anio'] = ((ultimo - historia[0]) / 2)[indices]
    for radio in (2, 3):
        kernel = np.ones((2 * radio + 1, 2 * radio + 1))
        kernel[1:-1, 1:-1] = 0
        suma = convolve(ultimo, kernel, mode='constant', cval=0)
        cantidad = convolve(np.ones_like(ultimo), kernel, mode='constant', cval=0)
        tabla[f'anillo_{radio}_ha'] = (suma / cantidad)[indices]
    return tabla

In [4]:
division = cfg['division_temporal']
auditoria = []
for rol, grupo in zonas.items():
    for zona in grupo:
        cubo = calcular_cubos(zona, celdas['bosque_min_pct'], celdas['celda_px'])
        anios = (range(division['entrenamiento'][0], division['validacion'][1] + 1)
                 if rol == 'entrenamiento' else division['prueba'])
        tablas = [tabla_espacial(cubo, anio - 2001) for anio in anios]
        panel = pd.concat(tablas, ignore_index=True).assign(zona=zona)
        assert not panel.duplicated(['anio_pred', 'celda_id']).any()
        assert (panel.anio_obs + 1 == panel.anio_pred).all()
        assert np.isfinite(panel.select_dtypes('number')).all().all()
        assert ('alerta' in panel) == (rol == 'entrenamiento')
        panel.to_csv(cache / f'{zona}.csv', index=False)
        auditoria.append([zona, rol, panel.anio_pred.min(), panel.anio_pred.max(), len(panel)])
display(pd.DataFrame(auditoria, columns=['zona','rol','desde','hasta','filas']))

,zona,rol,desde,hasta,filas
0,guaviare,entrenamiento,2004,2024,258732
1,macarena,entrenamiento,2004,2024,258556
2,chaira,retenidas,2025,2025,12290
3,putumayo,retenidas,2025,2025,12321
